# Standarisasi & Pembersihan Dataset ISPU Jakarta (2010-2025)

**Tujuan:** Menggabungkan 16 file CSV ISPU yang memiliki skema kolom berbeda menjadi satu DataFrame tunggal yang bersih dan siap untuk modeling.

**Dataset:** ISPU (Indeks Standar Pencemar Udara) Jakarta tahun 2010-2025

**Author:** Data Science Team  
**Date:** 2026-02-02

---

## Ringkasan Proses Pembersihan:

1. **Mapping Nama Kolom** - Menyatukan variasi nama kolom ke format standar
2. **Standardisasi Kode Stasiun** - Mengubah semua lokasi menjadi kode DKI1-DKI5
3. **Pembersihan Missing Values** - Mengganti marker missing ("---", "-", dll) dengan NaN
4. **Konversi Tipe Data** - Memastikan kolom numerik bertipe float
5. **Fix Format Tanggal** - Memperbaiki error tanggal (terutama tahun 2022 dan 2024/2025)
6. **Standardisasi Label Target** - Mengubah semua kategori ke UPPERCASE
7. **Validasi & Output** - Menghasilkan satu CSV gabungan yang bersih

In [7]:
import pandas as pd
import numpy as np
import glob
import os
import re
import warnings

warnings.filterwarnings('ignore')

In [8]:
# Configuration
ISPU_DIR = "../dataset/ISPU"
OUTPUT_FILE = "dataset/ispu_cleaned.csv"

STATION_MAPPING = {
    'kelapa gading': 'DKI2',
    'jagakarsa': 'DKI3',
    'lubang buaya': 'DKI4',
    'kebon jeruk': 'DKI5',
    'bunderan hi': 'DKI1',
    'bundaran hi': 'DKI1',
    'dki1': 'DKI1',
    'dki2': 'DKI2',
    'dki3': 'DKI3',
    'dki4': 'DKI4',
    'dki5': 'DKI5',
}

# Column name mapping (from various formats to standard names)
COLUMN_MAPPING = {
    'pm10': 'pm10', 'pm_10': 'pm10', 'pm_sepuluh': 'pm10',
    'pm25': 'pm25', 'pm_25': 'pm25', 'pm_duakomalima': 'pm25',
    'so2': 'so2', 'sulfur_dioksida': 'so2',
    'co': 'co', 'karbon_monoksida': 'co',
    'o3': 'o3', 'ozon': 'o3',
    'no2': 'no2', 'nitrogen_dioksida': 'no2',
    'max': 'max',
    'critical': 'critical_parameter', 'parameter_pencemar_kritis': 'critical_parameter',
    'categori': 'target_kategori', 'kategori': 'target_kategori',
    'tanggal': 'tanggal', 'stasiun': 'stasiun', 'lokasi_spku': 'stasiun', 'bulan': 'bulan',
}

VALID_CATEGORIES = ['BAIK', 'SEDANG', 'TIDAK SEHAT', 'SANGAT TIDAK SEHAT', 'BERBAHAYA']
MISSING_MARKERS = ['---', '-', 'TIDAK ADA DATA', '', 'nan', 'NaN', 'NULL']

## Helper Functions

Fungsi-fungsi untuk membersihkan dan standardisasi data.

In [9]:
def standardize_column_names(df):
    df.columns = df.columns.str.lower().str.strip()
    df = df.rename(columns=COLUMN_MAPPING)
    return df

def extract_station_code(station_value):
    if pd.isna(station_value): return None
    station_str = str(station_value).lower().strip()
    
    for code in ['dki1', 'dki2', 'dki3', 'dki4', 'dki5']:
        if code in station_str: return code.upper()
    
    for location, code in STATION_MAPPING.items():
        if location in station_str: return code
    return None

def clean_missing_values(value):
    if pd.isna(value): return np.nan
    return np.nan if str(value).strip() in MISSING_MARKERS else value

def convert_to_numeric(value):
    if pd.isna(value): return np.nan
    try:
        str_value = str(value).replace(',', '.').strip()
        numeric_match = re.search(r'-?\d+\.?\d*', str_value)
        return float(numeric_match.group()) if numeric_match else np.nan
    except:
        return np.nan

def fix_excel_date(date_value):
    if pd.isna(date_value): return date_value
    try:
        date_float = float(date_value)
        if 1 <= date_float <= 60000:
            return pd.Timestamp('1899-12-30') + pd.Timedelta(days=int(date_float))
    except:
        pass
    
    try:
        return pd.to_datetime(date_value)
    except:
        return date_value

def standardize_category(category):
    if pd.isna(category): return np.nan
    cat = str(category).upper().strip()
    return cat if cat in VALID_CATEGORIES else np.nan

## Main Cleaning Pipeline

Fungsi utama untuk memproses file-file CSV.

In [10]:
def load_and_clean_single_file(filepath):
    df = pd.read_csv(filepath, encoding='utf-8')
    df = standardize_column_names(df)
    
    # Handle composite date columns
    if {'bulan', 'tanggal', 'periode_data'}.issubset(df.columns):
        df['year'] = df['periode_data'].astype(str).str[:4].astype(int)
        df['month'] = df['bulan'].astype(int)
        df['day'] = df['tanggal'].astype(int)
        df['tanggal'] = pd.to_datetime(df[['year', 'month', 'day']], errors='coerce')
    
    if 'stasiun' in df.columns:
        df['stasiun'] = df['stasiun'].apply(extract_station_code)
    
    if 'tanggal' in df.columns:
        if '2022' in filepath:
            df['tanggal'] = df['tanggal'].apply(fix_excel_date)
        df['tanggal'] = pd.to_datetime(df['tanggal'], errors='coerce')
    
    pollutant_cols = ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2', 'max']
    for col in df.columns:
        df[col] = df[col].apply(clean_missing_values)
        if col in pollutant_cols:
            df[col] = df[col].apply(convert_to_numeric)
    
    if 'target_kategori' in df.columns:
        df['target_kategori'] = df['target_kategori'].apply(standardize_category)
    
    desired_cols = ['tanggal', 'stasiun', 'pm10', 'pm25', 'so2', 'co', 
                    'o3', 'no2', 'max', 'critical_parameter', 'target_kategori']
    
    return df[[c for c in desired_cols if c in df.columns]]

In [11]:
def combine_all_files():
    all_dfs = []
    for filepath in glob.glob(os.path.join(ISPU_DIR, "*.csv")):
        try:
            all_dfs.append(load_and_clean_single_file(filepath))
        except Exception:
            continue
    
    combined_df = pd.concat(all_dfs, ignore_index=True)
    
    desired_cols = ['tanggal', 'stasiun', 'pm10', 'pm25', 'so2', 'co', 
                    'o3', 'no2', 'max', 'critical_parameter', 'target_kategori']
    
    for col in desired_cols:
        if col not in combined_df.columns:
            combined_df[col] = np.nan
            
    return combined_df[desired_cols].sort_values(['tanggal', 'stasiun']).reset_index(drop=True)

In [12]:
# Main Pipeline Execution
combined_df = combine_all_files()

# Filter invalid rows (missing category or all pollutants missing)
combined_df = combined_df[combined_df['target_kategori'].notna()]
pollutant_cols = ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2']
cleaned_df = combined_df[~combined_df[pollutant_cols].isna().all(axis=1)]

cleaned_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
print(f"Dataset processed: {cleaned_df.shape[0]} rows saved to {OUTPUT_FILE}")

Dataset processed: 15412 rows saved to dataset/ispu_cleaned.csv


In [13]:
# Validation
print(f"Data Date Range: {cleaned_df['tanggal'].min()} - {cleaned_df['tanggal'].max()}")
print(f"\nMissing Values:\n{cleaned_df.isna().sum()}")
print(f"\nCategory Distribution:\n{cleaned_df['target_kategori'].value_counts()}")
cleaned_df.head()

Data Date Range: 2010-01-01 00:00:00 - 2025-08-31 00:00:00

Missing Values:
tanggal                327
stasiun                  2
pm10                   709
pm25                  8801
so2                    301
co                     208
o3                     345
no2                    312
max                      0
critical_parameter      46
target_kategori          0
dtype: int64

Category Distribution:
target_kategori
SEDANG                10448
TIDAK SEHAT            2421
BAIK                   2342
SANGAT TIDAK SEHAT      200
BERBAHAYA                 1
Name: count, dtype: int64


,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical_parameter,target_kategori
0,2010-01-01,DKI1,60.0,NaN,4.0,73.0,27.0,14.0,73.0,CO,SEDANG
5,2010-01-02,DKI1,32.0,NaN,2.0,16.0,33.0,9.0,33.0,O3,BAIK
10,2010-01-03,DKI1,27.0,NaN,2.0,19.0,20.0,9.0,27.0,PM10,BAIK
15,2010-01-04,DKI1,22.0,NaN,2.0,16.0,15.0,6.0,22.0,PM10,BAIK
20,2010-01-05,DKI1,25.0,NaN,2.0,17.0,15.0,8.0,25.0,PM10,BAIK


##  Data Cleaning Complete!

### Output:
- **File:** `dataset/ispu_cleaned.csv`
- **Rows:** (See output above)
- **Columns:** 11